# Évaluation canonique — performance et popularité

Ce notebook lit exactement les trois bundles de l’évaluation finale commune. Il présente P@5, P@10 et le **Hit Rate@5** — part des cas ayant au moins un film aimé dans le top 5. La variance décrit la dispersion entre les 921 cas d’un test unique ; la popularité est un diagnostic des recommandations.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "scripts" / "Models" / "eval_diagnostics.py").is_file()
)
sys.path.insert(0, str((ROOT / "scripts" / "Models").resolve()))
from eval_diagnostics import popularity_table, variance_table

bundle_specs = {
    "baseline": ("eval_baseline.json", "baseline_overlap_mean"),
    "Cluster": ("eval_cluster.json", "user_cluster_groups"),
    "UserCF": ("eval_cf.json", "user_user_cf"),
}
bundles = {
    label: json.loads((ROOT / "results" / filename).read_text(encoding="utf-8"))
    for label, (filename, _) in bundle_specs.items()
}

reference = bundles["baseline"]
for label, bundle in bundles.items():
    expected_method = bundle_specs[label][1]
    assert bundle["n_cases"] == reference["n_cases"] == 921
    assert bundle["random_state"] == reference["random_state"] == 42
    assert bundle["protocol"] == reference["protocol"]
    assert set(bundle["methods"]) == {expected_method}

methods = {
    label: {**bundles[label]["methods"][method], "_source": bundles[label]["source"]}
    for label, (_, method) in bundle_specs.items()
}
assert all(len(data["P@5"]) == 921 for data in methods.values())
print("3 bundles validés : même protocole, 921 cas, random_state=42, méthodes attendues.")


3 bundles validés : même protocole, 921 cas, random_state=42, méthodes attendues.


## P@5, P@10 et Hit Rate@5 — trois méthodes


In [2]:
variance_result = variance_table(methods)
assert set(variance_result["method"]) == set(bundle_specs)
assert "HitRate@5" in variance_result.columns
display(
    variance_result.style.format(
        {"P@5": "{:.4f}", "P@10": "{:.4f}", "HitRate@5": "{:.1%}"}
    )
)


,method,source,P@5,P@10,HitRate@5,var_P@5,écart-type,n_cases
0,UserCF,cf,0.0932,0.0853,37.6%,0.018051,0.134354,921
1,Cluster,cluster,0.0534,0.0481,22.7%,0.011491,0.107196,921
2,baseline,baseline,0.0343,0.0372,15.5%,0.006995,0.083639,921


## Popularité des top-5 — trois méthodes


In [3]:
film_n_ratings = pd.Series(
    {int(movie_id): int(count) for movie_id, count in reference["film_n_ratings"].items()},
    dtype=int,
)
assert not film_n_ratings.empty

popularity_result = popularity_table(methods, film_n_ratings)
assert set(popularity_result["method"]) == set(bundle_specs)
display(popularity_result)


,method,source,rang pop. médian,top 20,top 21–100,hors top 100
0,baseline,baseline,147.0,0.061672,0.408469,0.529859
1,Cluster,cluster,42.0,0.372856,0.293377,0.333768
2,UserCF,cf,10.0,0.786123,0.197357,0.016520


Cellule excédentaire neutralisée.
